# Satellite observations vs PPE ensembles: per pyroCb event and per region

1. **Per event**: SPEXone super-obs (AOD, AAOD, SSA at 550 nm, AE 440-670 nm) next to the ensemble median, 5-95 % range and control member of each PPE source, against days since onset. Events use the 3-hourly ECHAM-HAM `ybhatti` PPE only; the monthly ICON-HAM `hjia` PPE appears in the regional section. Plume animations: `pyro-animate <pcb_id>`.
2. **Per region** (Alaska, Canada, Northern Asia): AeroVal statistics (bias, NMB, MNMB, FGE, RMSE, R) of the model against SPEXone for the regional average, for background scenes and for pyroCb-affected scenes (within 500 km of an event during its window), with the spread across PPE members.

VIIRS joins once its super-obs exist with the same columns.

**Restricted output.** Everything here derives from the event list. Figures and tables go to `$PYRO_DATA_ROOT/derived/`; never commit this notebook with outputs (`tests/test_notebooks.py` refuses it). Clear outputs before `git add`.

Inputs (pipeline plan §1e-§1g):
- `pyro-match <events.csv> --model --max-hours-after 240` (3-hourly ybhatti only; events are not matched with monthly output)
- `pyro-regional <events.csv> --ppe ybhatti_pd_3h --ppe hjia_pd_2024_25_monthly --allow-coarse`

Kernel: the project venv (`uv sync --all-extras`, includes `ipykernel`) or the container (`scripts/apptainer_kernel.sh install`, kernel "pyro (apptainer)").

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from pyro import compare, regional
from pyro.envfile import load_env_file

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if (REPO / "config" / "local.env").exists():
    load_env_file(REPO / "config" / "local.env")
DATA = Path(os.environ.get("PYRO_DATA_ROOT", REPO / "data"))
MATCH_DIR = Path(os.environ.get("PYRO_MATCH_DIR", DATA / "derived" / "spexone_event_match"))
REGIONAL_DIR = Path(os.environ.get("PYRO_REGIONAL_DIR", DATA / "derived" / "regional_stats"))
OUT = Path(os.environ.get("PYRO_COMPARE_OUT", DATA / "derived" / "event_comparison"))

PPE_SOURCES = None  # e.g. ["ybhatti_pd_3h", "hjia_pd_2024_25_monthly"]; None = every source found
SAVE_FIGURES = True
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(16)

In [ ]:
cmp = compare.load(MATCH_DIR, PPE_SOURCES)
print(
    f"{cmp.summary.height} events in the table, {len(cmp.events)} with collocated super-obs; "
    f"PPE sources: {sorted(cmp.table['dataset'].unique().to_list()) if cmp.table.height else []}"
)

## Which events each PPE source covers

Number of AOD super-obs per event and source (empty = no collocation: onset outside the model period, or no SPEXone pixel within 500 km).

In [ ]:
cov = compare.coverage(cmp)
display(cov)

## All events: observed vs ensemble median

`log10_ratio` = mean log10(model median / observed) for AOD and AAOD; `frac_in_5_95` = share of super-obs inside the ensemble 5-95 % range (0.9 if the spread were calibrated).

In [ ]:
fig = compare.plot_overview(cmp.table)
plt.show()

m = compare.metrics(cmp.table)
display(
    m.group_by("dataset", "variable")
    .agg(
        pl.col("pcb_id").n_unique().alias("n_events"),
        pl.col("n_superobs").sum(),
        pl.col("bias").median().alias("median_bias"),
        pl.col("log10_ratio").median().alias("median_log10_ratio"),
        pl.col("frac_in_5_95").mean(),
    )
    .sort("dataset", "variable")
)

## One event

Left: SPEXone pixels in the event window within 1000 km (AOD 550), the matched fire position and the 150/500 km rings. Right: per variable, SPEXone super-obs (black) and each source's ensemble median with 5-95 % bars (x: control member).

In [ ]:
EVENT = cmp.events[0] if cmp.events else None  # or a pcb_id string
if EVENT is not None:
    fig = compare.plot_event(cmp, EVENT)
    plt.show()
    display(m.filter(pl.col("pcb_id") == EVENT))

## Every event to disk

One PNG per event plus the comparison table, metrics and coverage (restricted; `pyro-sync push event_comparison -r gdrive`).

In [ ]:
if SAVE_FIGURES and cmp.events:
    figdir = OUT / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    cmp.table.write_parquet(OUT / "comparison.parquet", compression="zstd")
    m.write_csv(OUT / "metrics.csv")
    cov.write_csv(OUT / "coverage.csv")
    for ev in cmp.events:
        fig = compare.plot_event(cmp, ev)
        fig.savefig(figdir / f"{ev}.png", dpi=110)
        plt.close(fig)
    print(f"{len(cmp.events)} event figures in {figdir}")

## Regional statistics: regional average vs background vs pyroCb scenes

From `pyro-regional`. Categories: `all` (regional average), `background`, `pyrocb`. Statistics of the ensemble median (bars) with the 5-95 % range of each statistic across members (whiskers). AeroVal definitions: NMB = sum(m - o) / sum(o), MNMB = 2/N sum((m - o)/(m + o)), FGE = 2/N sum(|m - o|/(m + o)). A pyroCb signal shows as a more negative NMB/MNMB and larger RMSE/FGE in `pyrocb` than in `background`, for AOD and AAOD in particular.

In [ ]:
if (REGIONAL_DIR / "stats.parquet").exists():
    rst = pl.read_parquet(REGIONAL_DIR / "stats.parquet")
    rsum = pl.read_parquet(REGIONAL_DIR / "member_summary.parquet")
    season = rst.filter((pl.col("period") == "season") & (pl.col("estimate") == "ensemble_median"))
    display(
        season.select(
            "dataset",
            "region",
            "variable",
            "category",
            "n",
            "mean_obs",
            "mean_mod",
            "bias",
            "nmb",
            "mnmb",
            "fge",
            "rmse",
            "r",
        ).sort("dataset", "region", "variable", "category")
    )
    fig = regional.plot_regions(rst, rsum, metrics=("nmb", "mnmb", "rmse", "r"))
    plt.show()
else:
    print(f"no regional statistics in {REGIONAL_DIR}; run pyro-regional first")

### pyroCb minus background, per month

Difference of each statistic between pyroCb-affected and background scenes (ensemble median), per region and month.

In [ ]:
if (REGIONAL_DIR / "stats.parquet").exists():
    keys = ["dataset", "region", "period", "variable"]
    med = rst.filter(pl.col("estimate") == "ensemble_median")
    diff = (
        med.filter(pl.col("category") == "pyrocb")
        .join(med.filter(pl.col("category") == "background"), on=keys, suffix="_bg")
        .select(
            *keys,
            "n",
            "n_bg",
            *[
                (pl.col(s) - pl.col(f"{s}_bg")).alias(f"d_{s}")
                for s in ("mean_obs", "bias", "nmb", "mnmb", "fge", "rmse", "r")
            ],
        )
        .sort(keys)
    )
    display(diff)